# Grid Line Detection — Interactive Evaluation Notebook

Use this notebook to inspect how well the agent reads grid lines from your floor plan PDF.
There is no ground truth — you judge the results visually against the rendered drawing.

**Restart & Run All** to evaluate from scratch.

## Section 0 — Setup

Set `PDF_PATH` to your floor plan file, then run all cells.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings('ignore')

# ── USER CONFIGURATION ────────────────────────────────────────
PDF_PATH = "path/to/your/floorplan.pdf"   # ← SET THIS
# ─────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Markdown
from PIL import Image

# Make sure the agent modules are importable
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

SEALION_ONLINE = True
try:
    import ollama_client
    import pdf_renderer
    import agent
    print("✓ All modules loaded.")
except RuntimeError as e:
    SEALION_ONLINE = False
    print(f"⚠ SEA-LION is offline: {e}")
    print("Vision sections will be skipped — fix Ollama and re-run.")

if not os.path.isfile(PDF_PATH):
    print(f"⚠ PDF not found: {PDF_PATH}")
    print("Update PDF_PATH in Section 0 and re-run.")

# Shared state collected as cells run
_state = {}
print("Setup complete.")

## Section 1 — Raw Floor Plan Preview

Render all pages and display them so you can confirm you have the right PDF.

In [ ]:
if not os.path.isfile(PDF_PATH):
    print("⚠ PDF not found — skipping Section 1.")
else:
    print("Rendering PDF at 150 DPI for preview...")
    preview_images = pdf_renderer.pdf_to_images(PDF_PATH, dpi=150)
    _state['preview_images'] = preview_images
    page_count = len(preview_images)
    print(f"Pages: {page_count}")

    for i, img_path in enumerate(preview_images):
        img = Image.open(img_path)
        w, h = img.size
        print(f"  Page {i+1}: {w} x {h} px")
        fig, ax = plt.subplots(figsize=(16, 12))
        ax.imshow(img)
        ax.set_title(f"Page {i+1} of {page_count}", fontsize=14)
        ax.axis('off')
        plt.tight_layout()
        plt.show()

## Section 2 — Initial Detection (Single Pass)

What SEA-LION sees on a first look, before any verification or margin scanning.

In [ ]:
if not SEALION_ONLINE or not os.path.isfile(PDF_PATH):
    print("⚠ Skipping Section 2 (SEA-LION offline or PDF missing).")
else:
    # Use the preview images from Section 1, or re-render at 200 DPI
    if 'preview_images' not in _state:
        _state['preview_images'] = pdf_renderer.pdf_to_images(PDF_PATH, dpi=200)

    # Select the primary floor plan page
    images_200 = pdf_renderer.pdf_to_images(PDF_PATH, dpi=200)
    _state['images_200'] = images_200
    floor_plan_image = agent._select_floor_plan_page(images_200, verbose=True)
    _state['floor_plan_image'] = floor_plan_image

    print("Running initial detection...")
    initial = agent.tool_detect_grid(floor_plan_image)
    _state['initial'] = initial

    fig, ax = plt.subplots(figsize=(16, 12))
    ax.imshow(Image.open(floor_plan_image))
    ax.set_title("Floor Plan — Initial Detection", fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    print("\n── Initial Detection Result ──────────────────────")
    print(json.dumps(initial, indent=2))

## Section 3 — Verification Pass

SEA-LION checks its own answer. Differences between passes reveal uncertainty.

In [ ]:
if not SEALION_ONLINE or 'initial' not in _state:
    print("⚠ Skipping Section 3 (SEA-LION offline or Section 2 not run).")
else:
    print("Running verification pass...")
    verified = agent.tool_verify(_state['floor_plan_image'], _state['initial'])
    _state['verified'] = verified

    initial = _state['initial']

    def _fmt(v):
        if isinstance(v, list):
            return ' '.join(str(x) for x in v)
        return str(v)

    fields = ['total_grid_lines', 'vertical_labels', 'horizontal_labels', 'confidence']
    print(f"\n{'Field':<25} {'Initial':<35} {'Verified':<35} {'Status'}")
    print("-" * 105)
    for f in fields:
        a = _fmt(initial.get(f))
        b = _fmt(verified.get(f))
        status = "✓ Matched" if a == b else "⚠ Changed"
        print(f"{f:<25} {a:<35} {b:<35} {status}")

    print(f"\nNotes (initial):  {initial.get('notes', '')}")
    print(f"Notes (verified): {verified.get('notes', '')}")

## Section 4 — Margin Scans

Crop each margin band and inspect what SEA-LION reads there individually.

In [ ]:
if not SEALION_ONLINE or 'floor_plan_image' not in _state:
    print("⚠ Skipping Section 4 (SEA-LION offline or floor plan image not available).")
else:
    from PIL import Image as _Image
    import tempfile

    img = _Image.open(_state['floor_plan_image'])
    w, h = img.size
    band = int(agent.MARGIN_FRACTION * min(w, h))

    boxes = {
        'top':    (0, 0, w, band),
        'bottom': (0, h - band, w, h),
        'left':   (0, 0, band, h),
        'right':  (w - band, 0, w, h),
    }

    margin_results = {}
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    ax_map = {'top': axes[0,0], 'bottom': axes[0,1], 'left': axes[1,0], 'right': axes[1,1]}

    for side in ('top', 'bottom', 'left', 'right'):
        print(f"Scanning {side} margin...")
        scan = agent.tool_zoom_margin(_state['floor_plan_image'], side)
        margin_results[side] = scan['labels']

        crop = img.crop(boxes[side])
        ax = ax_map[side]
        ax.imshow(crop)
        ax.set_title(f"{side.capitalize()} margin → {scan['labels']}", fontsize=11)
        ax.axis('off')

    plt.tight_layout()
    plt.show()
    _state['margin_results'] = margin_results

    # Summary table
    print("\n── Margin Label Summary ─────────────────────────")
    print(f"{'Margin':<10} {'Labels Found'}")
    print("-" * 50)
    for side in ('top', 'bottom', 'left', 'right'):
        labels = ' '.join(margin_results[side]) if margin_results[side] else '(none)'
        print(f"{side.capitalize():<10} {labels}")

    # Highlight inconsistencies between opposing margins
    print("\n── Opposing Margin Consistency ──────────────────")
    for pair in [('top', 'bottom'), ('left', 'right')]:
        a, b = pair
        la = sorted(margin_results.get(a, []))
        lb = sorted(margin_results.get(b, []))
        if la == lb:
            print(f"  {a}/{b}: ✓ consistent")
        else:
            diff = set(la).symmetric_difference(set(lb))
            print(f"  {a}/{b}: ⚠ differ → {sorted(diff)}")

## Section 5 — Full Agent Run

End-to-end run of the complete agentic workflow with verbose logging.

In [ ]:
if not SEALION_ONLINE or not os.path.isfile(PDF_PATH):
    print("⚠ Skipping Section 5 (SEA-LION offline or PDF missing).")
else:
    print("Running full agent...")
    print("=" * 60)
    full_result = agent.run(PDF_PATH, verbose=True)
    _state['full_result'] = full_result

    print("=" * 60)
    print("\n── Final JSON Result ─────────────────────────────")
    # Print without steps_taken for readability
    display_result = {k: v for k, v in full_result.items() if k != 'steps_taken'}
    print(json.dumps(display_result, indent=2))

    # Annotated image preview (if PIL can do it inline)
    try:
        images_200 = _state.get('images_200') or pdf_renderer.pdf_to_images(PDF_PATH, dpi=200)
        img = Image.open(images_200[0]).convert('RGB')
        from PIL import ImageDraw, ImageFont
        draw = ImageDraw.Draw(img)
        w, h = img.size
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 28)
        except IOError:
            font = ImageFont.load_default()

        vertical_labels = full_result.get('vertical_labels', [])
        horizontal_labels = full_result.get('horizontal_labels', [])

        if vertical_labels:
            n = len(vertical_labels)
            step = w // (n + 1)
            for i, lbl in enumerate(vertical_labels):
                x = step * (i + 1)
                draw.rectangle([x-20, 10, x+20, 44], fill=(255,255,0))
                draw.text((x-15, 12), lbl, fill=(0,0,0), font=font)

        if horizontal_labels:
            n = len(horizontal_labels)
            step = h // (n + 1)
            for i, lbl in enumerate(horizontal_labels):
                y = step * (i + 1)
                draw.rectangle([10, y-17, 50, y+17], fill=(0,200,255))
                draw.text((12, y-14), lbl, fill=(0,0,0), font=font)

        fig, ax = plt.subplots(figsize=(16, 12))
        ax.imshow(img)
        ax.set_title("Full Agent Run — Annotated Result", fontsize=14)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"(Annotation preview skipped: {e})")

## Section 6 — Detection Summary

Formatted summary table of all detection passes.

In [ ]:
if 'full_result' not in _state:
    print("⚠ Skipping Section 6 (run Section 5 first).")
else:
    fr = _state['full_result']
    init = _state.get('initial', {})
    ver  = _state.get('verified', {})
    mr   = _state.get('margin_results', {})

    v_labels = ' '.join(fr.get('vertical_labels', [])) or '(none)'
    h_labels = ' '.join(fr.get('horizontal_labels', [])) or '(none)'
    v_count  = len(fr.get('vertical_labels', []))
    h_count  = len(fr.get('horizontal_labels', []))
    total    = fr.get('total_grid_lines', 0)
    conf     = fr.get('confidence', 0)
    retries  = len([s for s in fr.get('steps_taken', []) if 'Attempt' in s])

    init_total = init.get('total_grid_lines', 'N/A')
    ver_total  = ver.get('total_grid_lines', 'N/A')

    if init_total == ver_total:
        ver_note = '(no change)'
    else:
        ver_note = f'(changed from {init_total})'

    margin_note = 'skipped (confident)' if not mr else 'completed'
    used_hint   = '✓ yes' if fr.get('used_reference_hint') else 'no'

    W = 56
    def row(label, value):
        label_w = 32
        value_w = W - label_w - 5
        return f"║ {label:<{label_w}} ║ {str(value):<{value_w}} ║"

    sep = '╠' + '═'*34 + '╬' + '═'*(W-34-1) + '╣'
    top = '╔' + '═'*34 + '╦' + '═'*(W-34-1) + '╗'
    bot = '╚' + '═'*34 + '╩' + '═'*(W-34-1) + '╝'
    title_line = '║' + ' GRID LINE DETECTION SUMMARY'.center(W) + '║'

    print(top)
    print(title_line)
    print(sep)
    print(row('Total grid lines detected', total))
    print(sep)
    print(row(f'Vertical lines  ({v_count})', f'→  {v_labels}'))
    print(row(f'Horizontal lines  ({h_count})', f'→  {h_labels}'))
    print(sep)
    print(row('Initial detection', init_total))
    print(row('After verification', f'{ver_total}  {ver_note}'))
    print(row('After margin scan', margin_note))
    print(sep)
    print(row('Agent confidence', f'{conf:.2f}'))
    print(row('Retries', retries))
    print(row('Used reference hint', used_hint))
    print(bot)

## Section 7 — Improvement Areas

Auto-generated findings based on the actual run results.

In [ ]:
if 'full_result' not in _state:
    print("⚠ Skipping Section 7 (run Section 5 first).")
else:
    fr   = _state['full_result']
    init = _state.get('initial', {})
    ver  = _state.get('verified', {})
    mr   = _state.get('margin_results', {})

    findings = []
    conf = float(fr.get('confidence', 0))

    # 1. Low confidence
    if conf < 0.85:
        findings.append(
            f"Confidence LOW ({conf:.2f}). Review margin scan results in Section 4. "
            "Consider increasing render DPI (try 300) for this floor plan."
        )

    # 2. Initial vs verified mismatch
    if init and ver:
        iv = sorted(init.get('vertical_labels', []))
        vv = sorted(ver.get('vertical_labels', []))
        ih = sorted(init.get('horizontal_labels', []))
        vh = sorted(ver.get('horizontal_labels', []))
        inconsistent = []
        if iv != vv:
            inconsistent += list(set(iv).symmetric_difference(set(vv)))
        if ih != vh:
            inconsistent += list(set(ih).symmetric_difference(set(vh)))
        if inconsistent:
            findings.append(
                f"SEA-LION changed its answer between initial and verification. "
                f"Inconsistent labels: {sorted(set(inconsistent))}. "
                "Try a higher DPI render or inspect those margin crops manually."
            )

    # 3. Opposing margins inconsistent
    if mr:
        for a, b in [('top', 'bottom'), ('left', 'right')]:
            la = sorted(mr.get(a, []))
            lb = sorted(mr.get(b, []))
            if la != lb:
                diff = sorted(set(la).symmetric_difference(set(lb)))
                findings.append(
                    f"Labels on the {a.upper()} margin differ from the {b.upper()} margin: {diff}. "
                    "May indicate non-standard conventions or partial labels. "
                    "Inspect Section 4 crops manually."
                )

    # 4. Uncertain labels ("?" suffix)
    uncertain = [
        lbl for lbl in
        fr.get('vertical_labels', []) + fr.get('horizontal_labels', [])
        if '?' in str(lbl)
    ]
    if uncertain:
        findings.append(
            f"{len(uncertain)} label(s) flagged as uncertain by SEA-LION: {uncertain}. "
            "Inspect the relevant margin crop in Section 4."
        )

    # 5. Multiple retries
    retries = len([s for s in fr.get('steps_taken', []) if 'Attempt' in s])
    if retries > 1:
        findings.append(
            f"Agent retried {retries} time(s) before reaching a result. "
            "Consider whether this PDF needs a higher DPI or a prompt adjustment."
        )

    # 6. Reference hint was used
    if fr.get('used_reference_hint'):
        findings.append(
            "Agent used the reference notebook hint (last resort). "
            "The floor plan may have unusual conventions — review the final labels carefully."
        )

    print("\n## Improvement Areas")
    print("=" * 60)
    if not findings:
        print("  ✓ High confidence detection. No improvement actions needed.")
    else:
        for finding in findings:
            print(f"  • {finding}")
            print()